# Error Analysis By Model and Dataset


In [61]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd

In [62]:
# Load both result files
bedrock_path = "../results/synthetic_groundedness_challenger_set_bedrock_results.csv"
llm_path = "../results/synthetic_groundedness_challenger_set_llm_results.csv"

bedrock_df = pd.read_csv(bedrock_path)
llm_df = pd.read_csv(llm_path)

# Merge on (query, context, response, label) which uniquely define a test case
merged_results = pd.merge(
    llm_df,
    bedrock_df,
    on=["query", "context", "response", "label"],
    how="inner",
    suffixes=("_llm", "_bedrock")
)

merged_results["label"] = merged_results["label"].str.upper().map({"PASS": 1, "FAIL": 0})
merged_results


,query,context,response,label,llm_score,llm_reasoning,bedrock_action,grounding_score,grounding_threshold,grounding_decision_reason,blocked_output_text
0,I made a big purchase that I regretted and can...,### Refund Not Showing Up? Here's What You Nee...,"To verify your refund, please follow these ste...",1,PASS,The ANSWER provides steps to verify the refund...,NONE,NaN,NaN,NaN,NaN
1,I am still waiting to find out the status of m...,**Refund Status Inquiry**\n\nWe understand tha...,We understand that waiting for a refund can be...,1,PASS,The ANSWER closely follows the steps outlined ...,NONE,NaN,NaN,NaN,NaN
2,"As advised by you, i requested seller to refun...",### Refund Status Inquiry\n\nThank you for rea...,Thank you for reaching out to us regarding you...,1,PASS,The ANSWER closely follows the guidance provid...,NONE,NaN,NaN,NaN,NaN
3,"Hurry and refund me, I am waiting",### Refund Not Showing Up\n\nWe understand tha...,We understand that waiting for a refund can be...,1,PASS,The ANSWER accurately reflects the information...,NONE,NaN,NaN,NaN,NaN
4,I was supposed to get a refund but I do not se...,**FAQ: My Refund is Not Showing Up**\n\nIf you...,If you are expecting a refund but do not see i...,1,PASS,The ANSWER provides steps to follow if a refun...,NONE,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
765,The wrong exchange rate was used for my cash w...,**Help & FAQs: Exchange Rates for Cash Withdra...,I apologize if you encountered any discrepanci...,0,PASS,The ANSWER acknowledges potential discrepancie...,GUARDRAIL_INTERVENED,NaN,NaN,NaN,"Sorry, the model cannot answer this question."
766,Can i get money in my home currency at my ATM ...,### Can I Withdraw Cash in My Home Currency at...,Can i get money in my home currency at my ATM?...,0,FAIL,The DOCUMENT states that ATMs do not offer the...,NONE,NaN,NaN,NaN,NaN
767,The exchange rate applied was incorrect when I...,**Exchange Rates for Cash Withdrawals While Tr...,I understand that you noticed the exchange rat...,0,FAIL,The DOCUMENT discusses exchange rates for cash...,GUARDRAIL_INTERVENED,NaN,NaN,NaN,"Sorry, the model cannot answer this question."
768,I exchanged cash when traveling and the rate w...,**Help Section: Currency Exchange Rates for Ca...,I'm sorry to hear that you experienced an issu...,0,FAIL,The ANSWER introduces the concept of a 'Platin...,GUARDRAIL_INTERVENED,NaN,NaN,NaN,"Sorry, the model cannot answer this question."


In [63]:
# Load the two files
example_path = "../results/retailbank_example_df.csv"
test_path = "../results/retailbankQA_test.csv"

example_df = pd.read_csv(example_path)
test_df = pd.read_csv(test_path)

example_df = example_df.drop(columns=["source_ds", "source"])
# Merge on a common key — assuming "test_case_id" in example_df corresponds to "id" in test_df
merged_df = pd.merge(example_df, test_df, left_on="test_case_id", right_on="id", how="left")
merged_df

,test_case_id,chunk_id,pred_prob,pred_label,max_pred_prob,example_pred_label,true_label,id,context,query,response,label,source_ds,source
0,retailbankQA_test_0,0_0,0.998359,1,0.998359,1,1,retailbankQA_test_0,### Refund Not Showing Up? Here's What You Nee...,I made a big purchase that I regretted and can...,"To verify your refund, please follow these ste...",1,retailbankQA_test,retailbankQA_test
1,retailbankQA_test_1,1_0,0.026392,0,0.994920,1,1,retailbankQA_test_1,**Refund Status Inquiry**\n\nWe understand tha...,I am still waiting to find out the status of m...,We understand that waiting for a refund can be...,1,retailbankQA_test,retailbankQA_test
2,retailbankQA_test_2,2_0,0.994479,1,0.994772,1,1,retailbankQA_test_2,### Refund Status Inquiry\n\nThank you for rea...,"As advised by you, i requested seller to refun...",Thank you for reaching out to us regarding you...,1,retailbankQA_test,retailbankQA_test
3,retailbankQA_test_3,3_0,0.994941,1,0.994941,1,1,retailbankQA_test_3,### Refund Not Showing Up\n\nWe understand tha...,"Hurry and refund me, I am waiting",We understand that waiting for a refund can be...,1,retailbankQA_test,retailbankQA_test
4,retailbankQA_test_4,4_0,0.998551,1,0.998551,1,1,retailbankQA_test_4,**FAQ: My Refund is Not Showing Up**\n\nIf you...,I was supposed to get a refund but I do not se...,If you are expecting a refund but do not see i...,1,retailbankQA_test,retailbankQA_test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
765,retailbankQA_test_765,765_0,0.000652,0,0.000717,0,0,retailbankQA_test_765,**Help & FAQs: Exchange Rates for Cash Withdra...,The wrong exchange rate was used for my cash w...,I apologize if you encountered any discrepanci...,0,retailbankQA_test,retailbankQA_test
766,retailbankQA_test_766,766_0,0.000688,0,0.000719,0,0,retailbankQA_test_766,### Can I Withdraw Cash in My Home Currency at...,Can i get money in my home currency at my ATM ...,Can i get money in my home currency at my ATM?...,0,retailbankQA_test,retailbankQA_test
767,retailbankQA_test_767,767_0,0.000529,0,0.000529,0,0,retailbankQA_test_767,**Exchange Rates for Cash Withdrawals While Tr...,The exchange rate applied was incorrect when I...,I understand that you noticed the exchange rat...,0,retailbankQA_test,retailbankQA_test
768,retailbankQA_test_768,768_0,0.000525,0,0.000525,0,0,retailbankQA_test_768,**Help Section: Currency Exchange Rates for Ca...,I exchanged cash when traveling and the rate w...,I'm sorry to hear that you experienced an issu...,0,retailbankQA_test,retailbankQA_test


In [64]:
# Ensure all join columns are strings
join_keys = ["query", "context", "response", "label"]

for key in join_keys:
    merged_df[key] = merged_df[key].astype(str)
    merged_results[key] = merged_results[key].astype(str)

# Now safely merge
final_merged = pd.merge(
    merged_df,
    merged_results,
    on=join_keys,
    how="left"
)

final_merged

# Select relevant columns
columns_to_keep = [
    "test_case_id",
    "query",
    "context",
    "response",
    "llm_score",
    "bedrock_action",
    "true_label",
    "example_pred_label",
    "source_ds",
    "source"
]
final_retail = final_merged[columns_to_keep]

# Rename for clarity
final_retail.columns = [
    "test_case_id", "query", "context", "response",
    "llm_score", "bedrock_action", "true_label", "cross_encoder_pred", "source_ds", "source"
]

# Convert outcome columns to binary flags
final_retail["llm_score"] = final_retail["llm_score"].str.upper().map({"PASS": 1, "FAIL": 0})
final_retail["bedrock_action"] = final_retail["bedrock_action"].apply(lambda x: 1 if x == "NONE" else 0)

final_retail


C:\Users\tahle\AppData\Local\Temp\ipykernel_24144\3843102854.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_retail["llm_score"] = final_retail["llm_score"].str.upper().map({"PASS": 1, "FAIL": 0})
C:\Users\tahle\AppData\Local\Temp\ipykernel_24144\3843102854.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_retail["bedrock_action"] = final_retail["bedrock_action"].apply(lambda x: 1 if x == "NONE" else 0)


,test_case_id,query,context,response,llm_score,bedrock_action,true_label,cross_encoder_pred,source_ds,source
0,retailbankQA_test_0,I made a big purchase that I regretted and can...,### Refund Not Showing Up? Here's What You Nee...,"To verify your refund, please follow these ste...",1,1,1,1,retailbankQA_test,retailbankQA_test
1,retailbankQA_test_1,I am still waiting to find out the status of m...,**Refund Status Inquiry**\n\nWe understand tha...,We understand that waiting for a refund can be...,1,1,1,1,retailbankQA_test,retailbankQA_test
2,retailbankQA_test_2,"As advised by you, i requested seller to refun...",### Refund Status Inquiry\n\nThank you for rea...,Thank you for reaching out to us regarding you...,1,1,1,1,retailbankQA_test,retailbankQA_test
3,retailbankQA_test_3,"Hurry and refund me, I am waiting",### Refund Not Showing Up\n\nWe understand tha...,We understand that waiting for a refund can be...,1,1,1,1,retailbankQA_test,retailbankQA_test
4,retailbankQA_test_4,I was supposed to get a refund but I do not se...,**FAQ: My Refund is Not Showing Up**\n\nIf you...,If you are expecting a refund but do not see i...,1,1,1,1,retailbankQA_test,retailbankQA_test
...,...,...,...,...,...,...,...,...,...,...
765,retailbankQA_test_765,The wrong exchange rate was used for my cash w...,**Help & FAQs: Exchange Rates for Cash Withdra...,I apologize if you encountered any discrepanci...,1,0,0,0,retailbankQA_test,retailbankQA_test
766,retailbankQA_test_766,Can i get money in my home currency at my ATM ...,### Can I Withdraw Cash in My Home Currency at...,Can i get money in my home currency at my ATM?...,0,1,0,0,retailbankQA_test,retailbankQA_test
767,retailbankQA_test_767,The exchange rate applied was incorrect when I...,**Exchange Rates for Cash Withdrawals While Tr...,I understand that you noticed the exchange rat...,0,0,0,0,retailbankQA_test,retailbankQA_test
768,retailbankQA_test_768,I exchanged cash when traveling and the rate w...,**Help Section: Currency Exchange Rates for Ca...,I'm sorry to hear that you experienced an issu...,0,0,0,0,retailbankQA_test,retailbankQA_test


In [65]:
# Load the latest uploaded files
llm_path = "../results/halubench_llm_results.csv"
bedrock_path = "../results/halubench_bedrock_results.csv"
truth_path = "../results/halubench_example_df.csv"

llm_df = pd.read_csv(llm_path)
bedrock_df = pd.read_csv(bedrock_path)
truth_df = pd.read_csv(truth_path)

# Standardize column names for merge
llm_df = llm_df.rename(columns={"id": "test_case_id"})
bedrock_df = bedrock_df.rename(columns={"id": "test_case_id"})

# Merge all three dataframes on 'test_case_id'
merged = llm_df.merge(bedrock_df, on="test_case_id", suffixes=("_llm", "_bedrock"))
merged = merged.merge(truth_df, on="test_case_id", suffixes=("", "_truth"))

merged


,test_case_id,context_llm,query_llm,response_llm,label_llm,source_ds_llm,source_llm,llm_score,llm_reasoning,context_bedrock,...,grounding_decision_reason,blocked_output_text,chunk_id,pred_prob,pred_label,max_pred_prob,example_pred_label,true_label,source_ds,source
0,26104577,The prognostic importance of extramedullary in...,Does waldenström macroglobulinemia with extram...,Yes. We show that extramedullary involvement a...,0,pubmedQA,halubench,FAIL,The DOCUMENT states that EMWM patients had a s...,The prognostic importance of extramedullary in...,...,NaN,"Sorry, the model cannot answer this question.",1935_0,0.965447,1,0.965447,1,0,pubmedQA,halubench
1,financebench_id_02656,128 \nConsolidated Statements of Operations\nY...,What is the FY2019 - FY2021 3 year average of ...,75.1%,0,FinanceBench,halubench,FAIL,To calculate the COGS as a percentage of reven...,128 \nConsolidated Statements of Operations\nY...,...,NaN,NaN,2020_0,0.699952,1,0.788797,1,0,FinanceBench,halubench
2,financebench_id_08319,PART II\nItem 8\n \n \nBALANCE SHEETS\n \n(In ...,We need to calculate a reasonable approximatio...,"$364,850.00",0,FinanceBench,halubench,FAIL,The DOCUMENT provides the total assets for Mic...,PART II\nItem 8\n \n \nBALANCE SHEETS\n \n(In ...,...,NaN,"Sorry, the model cannot answer this question.",2069_0,0.552437,1,0.552437,1,0,FinanceBench,halubench
3,12135,passage 1:1 Spray a baking sheet with butter f...,how to cook pollock?,"Based on the provided passages, here is how to...",1,RAGTruth,halubench,FAIL,The ANSWER accurately summarizes the steps fro...,passage 1:1 Spray a baking sheet with butter f...,...,NaN,NaN,3039_0,0.881065,1,0.983435,1,1,RAGTruth,halubench
4,21442579,Intra-aortic balloon pump (IABP) is an establi...,Does intra-aortic balloon pump implantation af...,No. This study demonstrates that CABG with IAB...,1,pubmedQA,halubench,PASS,The DOCUMENT provides data on the survival rat...,Intra-aortic balloon pump (IABP) is an establi...,...,NaN,"Sorry, the model cannot answer this question.",1228_0,0.062358,0,0.173395,0,1,pubmedQA,halubench
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
579,financebench_id_05629,Table of Contents\n \n \nintel corporation\nco...,By relying on the line items plainly stated wi...,5.7%,1,FinanceBench,halubench,PASS,The DOCUMENT provides Intel's net revenue for ...,Table of Contents\n \n \nintel corporation\nco...,...,NaN,"Sorry, the model cannot answer this question.",2782_0,0.522880,1,0.522880,1,1,FinanceBench,halubench
580,9555b283-c5a8-467a-b0bd-84ca6bdd54c0,Hoping to rebound from their home loss to the...,Who got the Cardinals their first points of th...,Anquan Boldin,0,DROP,halubench,FAIL,The DOCUMENT states that the Cardinals scored ...,Hoping to rebound from their home loss to the...,...,NaN,"Sorry, the model cannot answer this question.",293_0,0.585103,1,0.585103,1,0,DROP,halubench
581,financebench_id_05288,"SQUARE,INC.\nCONSOLIDATEDSTATEMENTSOFOPERATION...",We need to calculate a reasonable approximatio...,-4.5%,1,FinanceBench,halubench,PASS,The operating income (or loss) for each year i...,"SQUARE,INC.\nCONSOLIDATEDSTATEMENTSOFOPERATION...",...,NaN,"Sorry, the model cannot answer this question.",2868_0,0.624265,1,0.630159,1,1,FinanceBench,halubench
582,15838438,To determine which of the eye's refractive com...,Is high myopia associated with retinopathy of ...,Yes. High myopia associated with ROP appears p...,0,pubmedQA,halubench,FAIL,"The DOCUMENT states that in ROP eyes, increasi...",To determine which of the eye's refractive com...,...,NaN,"Sorry, the model cannot answer this question.",1612_0,0.939238,1,0.939238,1,0,pubmedQA,halubench


In [66]:
# Select relevant columns
columns_to_keep = [
    "test_case_id",
    "query_llm",
    "context_llm",
    "response_llm",
    "llm_score",
    "bedrock_action",
    "true_label",
    "example_pred_label",
    "source_ds_llm",
    "source_llm"
]
merged_filtered = merged[columns_to_keep]

# Rename for clarity
merged_filtered.columns = [
    "test_case_id", "query", "context", "response",
    "llm_score", "bedrock_action", "true_label", "cross_encoder_pred", "source_ds", "source"
]


# Convert outcome columns to binary flags
merged_filtered["llm_score"] = merged_filtered["llm_score"].str.upper().map({"PASS": 1, "FAIL": 0})
merged_filtered["bedrock_action"] = merged_filtered["bedrock_action"].apply(lambda x: 1 if x == "NONE" else 0)

merged_filtered
combined_df = pd.concat([merged_filtered, final_retail], ignore_index=True)
combined_df

C:\Users\tahle\AppData\Local\Temp\ipykernel_24144\358056414.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_filtered["llm_score"] = merged_filtered["llm_score"].str.upper().map({"PASS": 1, "FAIL": 0})
C:\Users\tahle\AppData\Local\Temp\ipykernel_24144\358056414.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_filtered["bedrock_action"] = merged_filtered["bedrock_action"].apply(lambda x: 1 if x == "NONE" else 0)


,test_case_id,query,context,response,llm_score,bedrock_action,true_label,cross_encoder_pred,source_ds,source
0,26104577,Does waldenström macroglobulinemia with extram...,The prognostic importance of extramedullary in...,Yes. We show that extramedullary involvement a...,0,0,0,1,pubmedQA,halubench
1,financebench_id_02656,What is the FY2019 - FY2021 3 year average of ...,128 \nConsolidated Statements of Operations\nY...,75.1%,0,1,0,1,FinanceBench,halubench
2,financebench_id_08319,We need to calculate a reasonable approximatio...,PART II\nItem 8\n \n \nBALANCE SHEETS\n \n(In ...,"$364,850.00",0,0,0,1,FinanceBench,halubench
3,12135,how to cook pollock?,passage 1:1 Spray a baking sheet with butter f...,"Based on the provided passages, here is how to...",0,1,1,1,RAGTruth,halubench
4,21442579,Does intra-aortic balloon pump implantation af...,Intra-aortic balloon pump (IABP) is an establi...,No. This study demonstrates that CABG with IAB...,1,0,1,0,pubmedQA,halubench
...,...,...,...,...,...,...,...,...,...,...
1349,retailbankQA_test_765,The wrong exchange rate was used for my cash w...,**Help & FAQs: Exchange Rates for Cash Withdra...,I apologize if you encountered any discrepanci...,1,0,0,0,retailbankQA_test,retailbankQA_test
1350,retailbankQA_test_766,Can i get money in my home currency at my ATM ...,### Can I Withdraw Cash in My Home Currency at...,Can i get money in my home currency at my ATM?...,0,1,0,0,retailbankQA_test,retailbankQA_test
1351,retailbankQA_test_767,The exchange rate applied was incorrect when I...,**Exchange Rates for Cash Withdrawals While Tr...,I understand that you noticed the exchange rat...,0,0,0,0,retailbankQA_test,retailbankQA_test
1352,retailbankQA_test_768,I exchanged cash when traveling and the rate w...,**Help Section: Currency Exchange Rates for Ca...,I'm sorry to hear that you experienced an issu...,0,0,0,0,retailbankQA_test,retailbankQA_test


In [67]:
# More succinct logic using DataFrame operations
pred_cols = ["cross_encoder_pred", "llm_score", "bedrock_action"]

# Count number of mismatches with true_label per row
combined_df["failures"] = (combined_df[pred_cols] != combined_df["true_label"].values.reshape(-1, 1)).sum(axis=1)

# Map to category labels
combined_df["match_category"] = combined_df["failures"].map({
    0: "all_match",
    1: "one_failure",
    2: "two_failures",
    3: "three_failures"
})

# Summarize results
match_summary_succinct = combined_df["match_category"].value_counts().reset_index()
match_summary_succinct.columns = ["category", "num_examples"]
match_summary_succinct


,category,num_examples
0,all_match,781
1,one_failure,317
2,two_failures,230
3,three_failures,26


In [68]:
combined_df.to_csv("combined_df_with_failures.csv", index=False)

In [70]:
combined_df

,test_case_id,query,context,response,llm_score,bedrock_action,true_label,cross_encoder_pred,source_ds,source,failures,match_category
0,26104577,Does waldenström macroglobulinemia with extram...,The prognostic importance of extramedullary in...,Yes. We show that extramedullary involvement a...,0,0,0,1,pubmedQA,halubench,1,one_failure
1,financebench_id_02656,What is the FY2019 - FY2021 3 year average of ...,128 \nConsolidated Statements of Operations\nY...,75.1%,0,1,0,1,FinanceBench,halubench,2,two_failures
2,financebench_id_08319,We need to calculate a reasonable approximatio...,PART II\nItem 8\n \n \nBALANCE SHEETS\n \n(In ...,"$364,850.00",0,0,0,1,FinanceBench,halubench,1,one_failure
3,12135,how to cook pollock?,passage 1:1 Spray a baking sheet with butter f...,"Based on the provided passages, here is how to...",0,1,1,1,RAGTruth,halubench,1,one_failure
4,21442579,Does intra-aortic balloon pump implantation af...,Intra-aortic balloon pump (IABP) is an establi...,No. This study demonstrates that CABG with IAB...,1,0,1,0,pubmedQA,halubench,2,two_failures
...,...,...,...,...,...,...,...,...,...,...,...,...
1349,retailbankQA_test_765,The wrong exchange rate was used for my cash w...,**Help & FAQs: Exchange Rates for Cash Withdra...,I apologize if you encountered any discrepanci...,1,0,0,0,retailbankQA_test,retailbankQA_test,1,one_failure
1350,retailbankQA_test_766,Can i get money in my home currency at my ATM ...,### Can I Withdraw Cash in My Home Currency at...,Can i get money in my home currency at my ATM?...,0,1,0,0,retailbankQA_test,retailbankQA_test,1,one_failure
1351,retailbankQA_test_767,The exchange rate applied was incorrect when I...,**Exchange Rates for Cash Withdrawals While Tr...,I understand that you noticed the exchange rat...,0,0,0,0,retailbankQA_test,retailbankQA_test,0,all_match
1352,retailbankQA_test_768,I exchanged cash when traveling and the rate w...,**Help Section: Currency Exchange Rates for Ca...,I'm sorry to hear that you experienced an issu...,0,0,0,0,retailbankQA_test,retailbankQA_test,0,all_match


In [75]:
import pandas as pd
from sklearn.metrics import classification_report

# Load the dataset
df = combined_df

# Define the three predicted columns and the true label column
pred_cols = ['llm_score', 'bedrock_action', 'cross_encoder_pred']
true_col = 'true_label'

# Convert values to string to avoid issues with classification report
df[true_col] = df[true_col].astype(str)
for col in pred_cols:
    df[col] = df[col].astype(str)

# Helper to generate classification reports
def generate_reports(group_name, group_df):
    results = {}
    for pred_col in pred_cols:
        report = classification_report(group_df[true_col], group_df[pred_col], output_dict=True, zero_division=0)
        results[f"{group_name} - {pred_col}"] = report
    return results

# 1. Whole dataset
overall_reports = generate_reports("Overall", df)

# 2. By 'source'
source_reports = {}
for source_name, group in df.groupby('source'):
    source_reports.update(generate_reports(f"Source: {source_name}", group))

# 3. By 'source_ds'
subdataset_reports = {}
for subds_name, group in df.groupby('source_ds'):
    subdataset_reports.update(generate_reports(f"Subdataset: {subds_name}", group))

# Combine all results into one dictionary
all_reports = {**overall_reports, **source_reports, **subdataset_reports}

# Convert to DataFrame for display
report_dfs = []
for name, report in all_reports.items():
    df_report = pd.DataFrame(report).transpose()
    df_report['Group'] = name
    report_dfs.append(df_report)

full_report_df = pd.concat(report_dfs).reset_index().rename(columns={'index': 'Class'})

#full_report_df.to_csv("evaluation_metrics.csv", index=False)

full_report_df


,Class,precision,recall,f1-score,support,Group
0,0,0.851711,0.706625,0.772414,634.000000,Overall - llm_score
1,1,0.775362,0.891667,0.829457,720.000000,Overall - llm_score
2,accuracy,0.805022,0.805022,0.805022,0.805022,Overall - llm_score
3,macro avg,0.813537,0.799146,0.800936,1354.000000,Overall - llm_score
4,weighted avg,0.811112,0.805022,0.802747,1354.000000,Overall - llm_score
...,...,...,...,...,...,...
115,0,0.989101,0.942857,0.965426,385.000000,Subdataset: retailbankQA_test - cross_encoder_...
116,1,0.945409,0.989610,0.967005,385.000000,Subdataset: retailbankQA_test - cross_encoder_...
117,accuracy,0.966234,0.966234,0.966234,0.966234,Subdataset: retailbankQA_test - cross_encoder_...
118,macro avg,0.967255,0.966234,0.966215,770.000000,Subdataset: retailbankQA_test - cross_encoder_...


### Run evaluation metrics on model outputs